# B-cell Epitope - phase 2 on Kaggle

Phase 1 is finished and the feature combination is frozen. This notebook does
one thing: fit each classifier on the whole B-cell set and score it on the SARS
set, which phase 1 never read.

Five configurations x 5 models = 25 runs:

| arm | configurations |
|---|---|
| the frozen combination | 1, read from `ML/frozen_configuration.json` |
| FS-1 Cihan et al. 2022 | 1, 6 columns as published |
| FS-2 Jain et al. 2021 | 1, 9 columns as published |
| FS-3 Gurcan et al. 2025 | 1, 9 columns as published |

Parts 2 to 5 of phase 1 do not run here. They are planned contrasts for the
internal table and were never candidates for the frozen configuration.

**SARS is a test set, not a selection set.** Nothing in this notebook chooses a
configuration, a threshold or a hyperparameter from a SARS number. The
threshold comes from the B-cell fit; the combination comes from phase 1.


In [ ]:
# ---- 1. Get the pipeline ----------------------------------------------------
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/azhar416/B-Cell-Epitopes.git"
PROJECT_DIR = Path("/kaggle/working/B-Cell-Epitopes")

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    if PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"{PROJECT_DIR} exists but holds no ML pipeline. Remove it, or point "
            "PROJECT_DIR at the repository root."
        )
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    raise FileNotFoundError(f"Clone finished but {PROJECT_DIR}/ML/train.py is absent.")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


def _git(*arguments):
    return subprocess.run(
        ["git", "-C", str(PROJECT_DIR), *arguments],
        capture_output=True, text=True,
    ).stdout.strip()


# A clone left behind by an earlier step would be reused silently and run old
# code, so confirm it is the current master rather than assuming it.
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "fetch", "-q", "--depth", "1", "origin", "master"],
    capture_output=True, text=True,
)
head = _git("rev-parse", "HEAD")
remote = _git("rev-parse", "FETCH_HEAD") or _git("rev-parse", "origin/master")

print(f"Pipeline at {PROJECT_DIR}")
print(f"  commit in use : {head[:12]}")
print(f"  origin/master : {remote[:12] or 'could not be read'}")
if remote and head != remote:
    raise RuntimeError(
        "This clone is not at origin/master, so it would run outdated code. "
        f"In use {head[:12]}, remote {remote[:12]}. Delete {PROJECT_DIR} and "
        "re-run this cell."
    )
print("  -> up to date")
COMMIT = head


In [ ]:
# ---- 2. Dependencies and imports -------------------------------------------
import importlib.util

required = ("numpy", "pandas", "sklearn", "imblearn", "joblib", "matplotlib", "xgboost")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print(f"Installing: {missing}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_DIR / "ML" / "requirements.txt")]
    )

import pandas as pd
from IPython.display import display, FileLink

from ML.CFG import CFG
from ML.collect_artifacts import collect_all
from ML.dataset import load_labeled_dataset
from ML.experiment_matrix import (
    MODEL_TYPES,
    build_experiment_matrix,
    report_environment,
    run_experiment_matrix,
)
from ML.summary_table import (
    build_table,
    read_summaries,
    render_pretty,
    render_tsv,
)

# scikit-learn decides the fold split and every estimator; pandas decides the
# row order feeding into it. Phase 2's rows sit in the same table as phase 1's,
# so a version change here would silently mix two protocols. The settings cell
# turns a mismatch into a hard stop.
ENVIRONMENT_OK = report_environment()
if not ENVIRONMENT_OK:
    print()
    print("Kaggle has refreshed its image since phase 1 ran. Pin the versions")
    print("with the line below, restart the kernel, and run again:")
    print(f"    !pip install -q -r {PROJECT_DIR}/ML/requirements.txt")


## Settings

One variable is worth setting by hand: `PHASE1_SUMMARY`, the path to phase 1's
`experiment_summary.csv` if you attached it as a Kaggle dataset. Without it the
SARS numbers are still produced, but the final table can only show the four
configurations this notebook ran instead of all 31 phase 1 measured.

The feature combination is **not** set by hand. It is read from the committed
`ML/frozen_configuration.json`.


In [ ]:
# ---- 3. Settings -----------------------------------------------------------
import json
from collections import Counter

SWEEP_MODELS = MODEL_TYPES

# Per-peptide prediction tables for 25 runs are small next to phase 1's 160.
COLLECT_PREDICTIONS = True

WORKING = Path("/kaggle/working")
RUN_DIR = WORKING / "ML-artifacts" / "phase2"
COLLECTED_DIR = WORKING / "collected" / "phase2"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = RUN_DIR / "experiment_summary.csv"

# Phase 1's summary, if you attached it as a Kaggle dataset. Optional.
PHASE1_SUMMARY = None   # Path("/kaggle/input/<dataset>/experiment_summary.csv")

# The combination is read from the committed selection record rather than typed
# here. Typing it by hand is how phase 2 ends up evaluating something the
# selection rule did not choose, with nothing in the output saying so.
FROZEN_PATH = PROJECT_DIR / "ML" / "frozen_configuration.json"
if not FROZEN_PATH.is_file():
    raise FileNotFoundError(
        f"{FROZEN_PATH} is absent. Run ML.select_configuration over phase 1's "
        "experiment_summary.csv, then commit and push the JSON it writes "
        "before running phase 2."
    )
frozen = json.loads(FROZEN_PATH.read_text(encoding="utf-8"))
FROZEN_GROUPS = tuple(frozen["selected_feature_groups"])
TUNING_METRIC = frozen["phase1_tuning_metric"]

# Phase 2's numbers are printed in the same table as phase 1's, so they have to
# come from the same image: scikit-learn decides the fold split and pandas the
# row order feeding it. This refuses rather than warns, because a mismatch here
# is not recoverable after the fact.
if not ENVIRONMENT_OK:
    raise RuntimeError(
        "This image is not the one phase 1 ran in, so phase 2 results could "
        "not be placed beside phase 1's. Pin the versions with "
        f"'!pip install -q -r {PROJECT_DIR}/ML/requirements.txt', restart the "
        "kernel, and run this notebook again."
    )
if not frozen.get("phase1_protocol_is_uniform", False):
    raise RuntimeError(
        "frozen_configuration.json reports that phase 1 did not run under one "
        "uniform protocol, so there is no single protocol for phase 2 to "
        "continue. Rerun phase 1 in one environment before continuing."
    )

# The repository ships the datasets, so prefer them; an attached Kaggle
# dataset is only a fallback.
def _find_data_dir(filename):
    bundled = PROJECT_DIR / "dataset"
    if (bundled / filename).is_file():
        return bundled
    found = sorted(Path("/kaggle/input").rglob(filename))
    if len(found) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}; found {len(found)}. "
            "Attach the dataset, or set DATA_DIR by hand."
        )
    return found[0].parent

DATA_DIR = _find_data_dir("input_bcell.csv")

cfg = CFG(
    DATASET_NAME="bcell",             # external_sars always trains on B-cell
    TRAINING_MODE="external_sars",    # phase 2: fit on B-cell, score on SARS
    DATA_DIR=DATA_DIR,
    OUTPUT_DIR=RUN_DIR,
    HYPERPARAMETER_TUNING_METRIC=TUNING_METRIC,
)
cfg.validate()

# Passing FROZEN_GROUPS restricts the proposed arm to that one combination and
# switches the planned contrasts off, leaving the frozen run plus the three
# published feature sets.
planned = build_experiment_matrix(SWEEP_MODELS, FROZEN_GROUPS)
arms = Counter(spec.feature_set for spec in planned)
papers = sum(count for name, count in arms.items() if name.startswith("perbandingan"))

print(f"frozen combination : {frozen['selected_label']} "
      f"({frozen['selected_n_features']} features)")
print(f"  groups           : {' '.join(FROZEN_GROUPS)}")
print(f"  rule             : {frozen['rule']}, metric {frozen['binding_metric']}")
print(f"  mean / threshold : {frozen['selected_mean']:.4f} / "
      f"{frozen['one_se_threshold']:.4f}")
print(f"  frozen at        : {frozen['frozen_at']}")
print()
print(f"models: {list(SWEEP_MODELS)}")
print(f"{len(planned)} runs = {len(planned) // len(SWEEP_MODELS)} configurations x "
      f"{len(SWEEP_MODELS)} models")
print(f"  frozen      {arms['proposed'] // len(SWEEP_MODELS):3d}  the combination phase 1 chose")
print(f"  papers      {papers // len(SWEEP_MODELS):3d}  FS-1/FS-1b/FS-2/FS-3 as published")
print(f"  contrasts   {arms['kontras'] // len(SWEEP_MODELS):3d}  parts 2-5 do not run in phase 2")
print()
print(f"mode={cfg.TRAINING_MODE} (SARS IS read, once, as a test set)")
print(f"tuning={cfg.HYPERPARAMETER_TUNING} metric={cfg.HYPERPARAMETER_TUNING_METRIC} "
      f"candidates/model={cfg.TUNING_CANDIDATES_PER_MODEL}")
print(f"class_weight={cfg.CLASS_WEIGHT} scale_pos_weight={cfg.XGBOOST_SCALE_POS_WEIGHT} "
      f"smote={cfg.USE_SMOTE} seed={cfg.RANDOM_STATE}")
print(f"flank window={cfg.FLANK_WINDOW} split_sides={cfg.FLANK_SPLIT_SIDES}")
print(f"data={DATA_DIR}")
print(f"out={RUN_DIR}")

expected_runs = 5 * len(SWEEP_MODELS)
if len(planned) != expected_runs:
    raise RuntimeError(
        f"Expected {expected_runs} runs (frozen + 4 papers, per model) but the "
        f"matrix planned {len(planned)}. Do not start the sweep until the plan "
        "is what you meant to run."
    )
# select_configuration reads phase 1's summary as CSV, so every protocol field
# in the record is a string. Compare as integers, or "42" != 42 stops a run
# that is in fact identical.
phase1_seed = int(frozen["phase1_random_state"])
if cfg.RANDOM_STATE != phase1_seed:
    raise RuntimeError(
        f"Seed differs from phase 1: {cfg.RANDOM_STATE} here against "
        f"{phase1_seed} then."
    )
print()
print(
    "Plan matches: frozen combination + 4 published feature sets, "
    f"{len(SWEEP_MODELS)} models."
)


## Restarting a run that ran out of time

Phase 2 is 25 runs on 14,323 training rows with no external cross-validation,
so it is unlikely to hit the session limit. This cell is here for symmetry with
phase 1; normally skip it.


In [ ]:
# ---- 4. Restart an interrupted session (normally skip) ---------------------
import shutil

# Example, when continuing a session that timed out:
#   PREVIOUS_OUTPUT = "/kaggle/input/bcell-phase2/ML-artifacts/phase2"
#   RESUME = True
PREVIOUS_OUTPUT = None
RESUME = False

if PREVIOUS_OUTPUT is not None:
    source = Path(PREVIOUS_OUTPUT)
    if not source.is_dir():
        raise FileNotFoundError(f"PREVIOUS_OUTPUT is not a directory: {source}")
    copied = 0
    for item in source.rglob("*"):
        if not item.is_file():
            continue
        target = RUN_DIR / item.relative_to(source)
        if target.exists():
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, target)
        copied += 1
    print(f"Restored {copied} file(s) into {RUN_DIR}")
    if not RESUME:
        print("WARNING: RESUME is False, so the restored runs will be repeated.")
elif RESUME:
    if not list(RUN_DIR.rglob("*_metrics.json")):
        print("WARNING: RESUME is True but RUN_DIR holds no reports, so nothing "
              "will be skipped. Set PREVIOUS_OUTPUT to the attached run directory.")
else:
    print("Fresh run; nothing to restore.")


## Check both datasets before spending time on them

Phase 1 only had to check B-cell. Phase 2 reads SARS too, and SARS is the one
set whose numbers cannot be re-earned if something is wrong with it: once it
has been looked at and a decision made from it, it is no longer external.

The overlap count must be zero. Cleaning removes every B-cell peptide that also appears in SARS, so nothing scored here was ever trained on; anything other than zero means that removal broke.


In [ ]:
# ---- 5. Data sanity check --------------------------------------------------
bcell = load_labeled_dataset(
    "bcell", cfg.DATA_DIR,
    max_peptide_length=cfg.MAX_PEPTIDE_LENGTH, feature_set="proposed",
)
sars = load_labeled_dataset(
    "sars", cfg.DATA_DIR,
    max_peptide_length=cfg.MAX_PEPTIDE_LENGTH, feature_set="proposed",
)

problems = []
for name, frame, expected in (
    ("B-cell", bcell, (14323, 758, {0: 10453, 1: 3870})),
    ("SARS", sars, (467, 1, {0: 364, 1: 103})),
):
    counts = frame.target.value_counts().sort_index().to_dict()
    proteins = int(frame.parent_protein_id.nunique())
    actual = (len(frame), proteins, counts)
    print(f"{name:<8} {len(frame):>6,} rows, {proteins:>4,} proteins, classes {counts}")
    if actual != expected:
        problems.append(f"{name}: expected {expected} but got {actual}")

print()
if problems:
    for problem in problems:
        print(f"WARNING: {problem}")
    print("The input files differ from the ones phase 1 ran on, so phase 2")
    print("numbers would not belong in the same table.")
else:
    print("Both match the reference cleaning counts.")

# A peptide present in both sets is not external for that row. This is a
# property of the published data, not something this pipeline can fix, so it is
# reported rather than removed -- dropping these rows would make the SARS
# numbers incomparable with the three papers, which all keep them.
overlap = sorted(set(sars.peptide_seq) & set(bcell.peptide_seq))
share = 100.0 * len(overlap) / len(sars)
print()
print(f"{len(overlap)} of {len(sars)} SARS peptides ({share:.1f}%) also appear "
      "verbatim in the B-cell training set.")
# Zero is now a requirement, not an observation: cleaning removes every
# B-cell peptide that also appears in SARS, so anything else means that
# removal broke and this would not be an external evaluation.
if overlap:
    raise RuntimeError(
        f"{len(overlap)} SARS peptide(s) are also in the training set. "
        "ML.dataset.load_labeled_dataset is where they are removed."
    )

# parent_protein_id is a label-encoded column. SARS holds one protein, and it
# is not in the B-cell set, so for every SARS row it encodes to the unseen
# marker. Any feature set carrying it loses that column entirely here.
sars_proteins = set(sars.parent_protein_id)
unseen = sars_proteins - set(bcell.parent_protein_id)
print()
print(f"SARS proteins: {sorted(sars_proteins)}")
print(f"  unseen in B-cell training: {sorted(unseen)}")
print("  -> parent_protein_id is constant across all SARS rows, so FS-3 loses")
print("     that column, and isoelectric_point, hydrophobicity and")
print("     protein_length are constant too: one protein, one value each.")


## Run

25 runs. Each fits the full B-cell set once per tuning candidate to pick
hyperparameters, refits on everything, and scores SARS. Expect well under an
hour.


In [ ]:
# ---- 6. Run ----------------------------------------------------------------
import time

started = time.time()
matrix_rows = run_experiment_matrix(
    RUN_DIR,
    base_config=cfg,
    model_types=SWEEP_MODELS,
    feature_groups=FROZEN_GROUPS,
    tuning_metric=TUNING_METRIC,
    resume=RESUME,
)
elapsed = time.time() - started

matrix_summary = pd.DataFrame(matrix_rows)
ok = int((matrix_summary["status"] == "success").sum())
print()
print(f"{len(matrix_summary)} runs in {elapsed / 3600:.2f} h -- "
      f"{ok} successful, {len(matrix_summary) - ok} failed")

failures = matrix_summary[matrix_summary["status"] != "success"]
if not failures.empty:
    display(failures[["experiment_id", "error"]])
display(FileLink(str(SUMMARY_PATH)))


## Collect every measurement

Same as phase 1: everything that is a measurement is rewritten into a few
long-format tables, because `/kaggle/working` is gone once the session ends.
`external_sars_predictions.csv` exists only in phase 2.


In [ ]:
# ---- 7. Collect metrics ----------------------------------------------------
manifest = collect_all(
    [SUMMARY_PATH], COLLECTED_DIR, include_predictions=COLLECT_PREDICTIONS
)

print(f"Collected {manifest['runs_collected']} run(s) into {COLLECTED_DIR}\n")
for name, info in manifest["tables"].items():
    size_mb = (COLLECTED_DIR / info["file"]).stat().st_size / 1048576
    print(f"  {name:<28} {info['rows']:>9,} rows  {size_mb:>7.1f} MB  {info['file']}")
    display(FileLink(str(COLLECTED_DIR / info["file"])))

if manifest["runs_missing_report"]:
    print(f"\nNo readable report for {len(manifest['runs_missing_report'])} run(s):")
    for experiment_id in manifest["runs_missing_report"]:
        print(f"  {experiment_id}")

display(FileLink(str(COLLECTED_DIR / "manifest.json")))


## The publication table

Internal columns are `mean +/- std` over the five protein-grouped folds;
external columns are single point estimates, because SARS is evaluated once.
The `Ext?` column says whether a row has SARS numbers at all.

With `PHASE1_SUMMARY` set, phase 1's internal measurements fill in beside phase
2's external ones for every row phase 1 produced. Without it, only the four
configurations this notebook ran appear.


In [ ]:
# ---- 8. Publication table --------------------------------------------------
summary_paths = [SUMMARY_PATH]
if PHASE1_SUMMARY is not None:
    phase1_path = Path(PHASE1_SUMMARY)
    if not phase1_path.is_file():
        raise FileNotFoundError(f"PHASE1_SUMMARY is not a file: {phase1_path}")
    # Phase 1 first: read_summaries lets later files win per populated field, so
    # phase 2's external columns are added without overwriting phase 1's
    # internal ones for the rows phase 2 did not rerun.
    summary_paths = [phase1_path, SUMMARY_PATH]
else:
    print("PHASE1_SUMMARY is not set, so this table shows only the four")
    print("configurations phase 2 ran. Attach phase 1's experiment_summary.csv")
    print("to get all 31 rows with their internal metrics.")
    print()

table_rows = build_table(read_summaries(summary_paths), model_order=SWEEP_MODELS)
print(render_pretty(table_rows))

TSV_PATH = COLLECTED_DIR / "publication_table.tsv"
TSV_PATH.write_text(render_tsv(table_rows), encoding="utf-8")
external = sum(1 for row in table_rows if row[10] == "TRUE")
print()
print(f"{external}/{len(table_rows)} rows carry SARS numbers.")
print("Download this and paste it into the spreadsheet:")
display(FileLink(str(TSV_PATH)))


## Drop the model bundles

Phase 2's bundles are the ones that would actually be used to predict new
peptides, so consider keeping the frozen combination's four before purging. Set
`DROP_MODEL_BUNDLES = False` to keep everything.


In [ ]:
# ---- 9. Reclaim space ------------------------------------------------------
DROP_MODEL_BUNDLES = True
DROP_PLOTS = False          # per-run PNGs; keep them if you want the curves

def _purge(pattern):
    files = list(RUN_DIR.rglob(pattern))
    freed = sum(path.stat().st_size for path in files)
    for path in files:
        path.unlink()
    return len(files), freed / 1048576

def _tree_size(root):
    return sum(p.stat().st_size for p in root.rglob("*") if p.is_file()) / 1048576

if DROP_MODEL_BUNDLES:
    # The frozen arm lives under RUN_DIR/proposed/<model>/<groups>/, so select
    # it by directory rather than by matching the group names into the path.
    keep = sorted((RUN_DIR / "proposed").rglob("*.joblib"))
    if keep:
        print(f"{len(keep)} bundle(s) for the frozen combination are about to go:")
        for path in keep:
            print(f"  {path.relative_to(RUN_DIR)}")
            display(FileLink(str(path)))
        print("Download them from the links above first if you want them.")
        print()
    else:
        print("No bundle found for the frozen combination; nothing to keep.")
        print()

print(f"Before: {_tree_size(RUN_DIR):,.1f} MB in {RUN_DIR}")
if DROP_MODEL_BUNDLES:
    count, freed = _purge("*.joblib")
    print(f"  removed {count} model bundles ({freed:,.1f} MB)")
if DROP_PLOTS:
    count, freed = _purge("*.png")
    print(f"  removed {count} plots ({freed:,.1f} MB)")
print(f"After:  {_tree_size(RUN_DIR):,.1f} MB")
print(f"Collected tables: {_tree_size(COLLECTED_DIR):,.1f} MB in {COLLECTED_DIR}")


## Before closing this session

1. **Save Version**, so Kaggle keeps the output.
2. Download from the links above:
   - `publication_table.tsv` -- the table to paste into the spreadsheet
   - `experiment_summary.csv` -- every field, including the protocol columns
   - `external_sars_predictions.csv` -- per-peptide SARS predictions, the only
     way to compute anything else about SARS later without rerunning
   - `fold_metrics.csv` -- per-fold internal values, needed for any paired test
   - `oof_predictions.csv.gz` -- per-peptide internal predictions

`experiment_summary.csv` and `fold_metrics.csv` only hold aggregates once the
session is gone, so the two prediction files are the ones that cannot be
reconstructed.

Phase 2 is the last training step. Nothing after this point should change a
feature set, a hyperparameter or a threshold.


## What these numbers do and do not support

**They support** a statement about transfer to an unseen protein: every model
was fitted without ever seeing `AAU93319`, and the decision threshold came from
the B-cell fit, not from SARS.

**They do not support** a claim that one configuration is better than another
*because* its SARS number is higher. SARS is 467 rows from a single protein, so
a difference of a few points is one protein's worth of noise, not evidence. The
configuration was chosen in phase 1, under the one-standard-error rule, from
internal protein-grouped cross-validation only. That ordering is the whole
reason these numbers mean anything.

**Read with care:**

- `parent_protein_id` encodes to the unseen marker for all 467 rows, and
  `isoelectric_point`, `hydrophobicity` and `protein_length` are constant
  across them. FS-3 and FS-2 therefore run on fewer varying columns here than
  they did internally. That is a real property of applying them to one
  protein, not a handicap this notebook imposes.
- No SARS peptide appears in the training set. The 35 that once did were removed from the development side, leaving SARS whole at 467 rows.
  Those rows are not external. The three papers keep them, so this pipeline
  keeps them too, and the share is reported rather than hidden.
- External metrics are point estimates with no spread. Do not write them as
  `mean +/- std`; there is only one evaluation.
